In [ ]:
!pip install torch transformers accelerate datasets scikit-learn numpy evidently matplotlib shap

import argparse
import torch
import numpy as np
import shap
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
import matplotlib.pyplot as plt
import os
import random


#################################
# Parse Arguments
#################################
def parse_args():
    parser = argparse.ArgumentParser(
        description="Run CLS token drift detection experiments with a true domain shift, now using SHAP."
    )
    parser.add_argument(
        "--model_name",
        type=str,
        default="nlpaueb/sec-bert-base",
        help="HuggingFace model name (SEC-BERT variant).",
    )
    parser.add_argument(
        "--wiki_dataset_name",
        type=str,
        default="wikitext",
        help="HuggingFace dataset for original text.",
    )
    parser.add_argument(
        "--wiki_dataset_config",
        type=str,
        default="wikitext-2-raw-v1",
        help="Dataset config.",
    )
    parser.add_argument(
        "--wiki_split", type=str, default="train", help="WikiText dataset split."
    )
    parser.add_argument(
        "--financial_dataset_name",
        type=str,
        default="financial_phrasebank",
        help="HuggingFace dataset for financial domain.",
    )
    parser.add_argument(
        "--financial_dataset_config",
        type=str,
        default="sentences_50agree",
        help="Configuration of financial_phrasebank.",
    )
    parser.add_argument(
        "--financial_split", type=str, default="train", help="Financial dataset split."
    )
    parser.add_argument(
        "--max_texts",
        type=int,
        default=30000,
        help="Max number of texts to use from dataset.",
    )
    parser.add_argument("--batch_size", type=int, default=64, help="Batch size.")
    parser.add_argument(
        "--alpha",
        type=float,
        default=0.01,
        help="(Not used anymore, kept for consistency)",
    )
    parser.add_argument(
        "--drift_threshold_std",
        type=float,
        default=3.0,
        help="Number of std devs for threshold.",
    )
    parser.add_argument(
        "--window_size",
        type=int,
        default=50,
        help="Rolling window size for threshold computation.",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default="results",
        help="Directory to save results and plots.",
    )
    # If running in a Jupyter/Colab environment, return default args
    args, unknown = parser.parse_known_args()
    return args


args = parse_args()

#################################
# Setup
#################################
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)

# Ensure output directory exists
os.makedirs(args.output_dir, exist_ok=True)

#################################
# Load Model and Tokenizer
#################################
print("Loading model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(args.model_name)
model = AutoModelForSequenceClassification.from_pretrained(args.model_name)
model.to(device)
model.eval()
print("Model ready on device:", device)

#################################
# Load Original (WikiText) Dataset
#################################
wiki_dataset = load_dataset(
    args.wiki_dataset_name, args.wiki_dataset_config, split=args.wiki_split
)
texts = wiki_dataset["text"]

if args.max_texts > 0 and args.max_texts < len(texts):
    texts = texts[: args.max_texts]
print(f"Original dataset loaded: {len(texts)} WikiText samples")

#################################
# Load Financial Dataset for Drift
#################################
fin_dataset = load_dataset(
    args.financial_dataset_name,
    args.financial_dataset_config,
    split=args.financial_split,
)
financial_texts = fin_dataset["sentence"]
random.shuffle(financial_texts)

# Simulate a second domain: "Kaggle-like" data (just a slice of finance here)
kaggle_texts = financial_texts[:1000]
print(f"Financial dataset loaded: {len(financial_texts)} samples")
print(f"Kaggle-like dataset loaded: {len(kaggle_texts)} samples")

#################################
# Simulate Multiple True Domain Drifts
#################################
n = len(texts)
# First drift region: from 1/3 to 1/2 (Finance)
drift_start_1 = n // 3
drift_end_1 = n // 2

# Second drift region: from 2/3 to 5/6 (Kaggle)
drift_start_2 = 2 * n // 3
drift_end_2 = 5 * n // 6

fin_idx = 0
for i in range(drift_start_1, drift_end_1):
    if fin_idx >= len(financial_texts):
        fin_idx = 0
    texts[i] = financial_texts[fin_idx]
    fin_idx += 1

kaggle_idx = 0
for i in range(drift_start_2, drift_end_2):
    if kaggle_idx >= len(kaggle_texts):
        kaggle_idx = 0
    texts[i] = kaggle_texts[kaggle_idx]
    kaggle_idx += 1

print("Simulated multiple domain drifts:")
print(f"  Drift region 1: indices {drift_start_1} to {drift_end_1} (Finance).")
print(f"  Drift region 2: indices {drift_start_2} to {drift_end_2} (Kaggle).")


#################################
# Utility Functions
#################################
def batch_generator(data, batch_size=32):
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]


#################################
# SHAP Helper: we need a prediction function
#################################
def shap_predict(texts_for_shap):
    """
    Return the model's logits (or probabilities) for SHAP to explain.
    We'll use logits for simplicity.
    """
    encodings = tokenizer(
        texts_for_shap,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        # shape = [batch_size, num_labels]
        logits = outputs.logits
    # Return as a CPU numpy array
    return logits.detach().cpu().numpy()


# Create a small background sample for SHAP from the original domain (WikiText)
# Keep it small so shap.Explainer doesn't blow up in memory/time.
background_size = 20
background_texts = texts[:background_size]

print("Creating SHAP explainer (this may be slow for large models/datasets)...")
explainer = shap.Explainer(shap_predict, background_texts)

#################################
# Main Experiment Loop (SHAP-based Drift)
#################################
print("\nStarting SHAP-based drift detection...")

shap_scores = []
thresholds = []
drift_points = []

for batch_index, batch_texts in enumerate(
    batch_generator(texts, batch_size=args.batch_size)
):
    # 1) Compute SHAP values for current batch
    #    The output will be a list (or array) of shape [batch_size, tokens, num_labels],
    #    or for some Explainer types, it might be [batch_size, num_labels].
    #    It depends on the Explainer. We need the .values attribute for shap.Explanation objects.
    #
    #    If the default is a token-level explanation, we'll just take a mean over all tokens & labels
    #    to get a single scalar for the batch. If using a different shap.Explainer, adjust accordingly.
    #
    explanation = explainer(batch_texts)
    # Explanation.values could be [batch_size, #tokens, #labels], depending on the explainer
    # We'll compute a simple average magnitude across all tokens and labels:
    current_shap_values = (
        explanation.values
    )  # shape: (batch_size, n_tokens, n_labels) in many cases
    abs_vals = np.abs(current_shap_values)
    # We'll get the mean across tokens & labels per sample, then average across the batch:
    sample_means = abs_vals.mean(axis=(1, 2))  # shape: (batch_size,)
    batch_score = sample_means.mean()  # single float for this batch

    shap_scores.append(batch_score)

    # 2) Compute rolling threshold if we have enough history
    if len(shap_scores) > args.window_size:
        recent_scores = shap_scores[-args.window_size :]
        mean_score = np.mean(recent_scores)
        std_score = np.std(recent_scores)
        threshold = mean_score + args.drift_threshold_std * std_score
        thresholds.append(threshold)

        if batch_score > threshold:
            print(
                f"Batch {batch_index}: Drift detected! SHAP score = {batch_score:.4f}, threshold = {threshold:.4f}"
            )
            print(f"Texts in this batch:\n{batch_texts}\n")
            drift_points.append(batch_index)
    else:
        thresholds.append(float("inf"))

#################################
# Save Results
#################################
shap_scores_path = os.path.join(args.output_dir, "shap_scores.npy")
thresholds_path = os.path.join(args.output_dir, "shap_thresholds.npy")
drift_points_path = os.path.join(args.output_dir, "shap_drift_points.npy")

np.save(shap_scores_path, shap_scores)
np.save(thresholds_path, thresholds)
np.save(drift_points_path, drift_points)

print(f"\nSHAP scores saved to {shap_scores_path}")
print(f"Thresholds saved to {thresholds_path}")
print(f"Drift points saved to {drift_points_path}")

#################################
# Compute FPR, TPR, Latency
#################################
num_batches = len(shap_scores)
gt_labels = np.zeros(num_batches, dtype=int)  # 0 = no drift, 1 = drift


def mark_drift_region(start_idx, end_idx):
    start_batch = start_idx // args.batch_size
    end_batch = (end_idx - 1) // args.batch_size
    for b in range(start_batch, end_batch + 1):
        gt_labels[b] = 1


# Mark the drift regions
mark_drift_region(drift_start_1, drift_end_1)
mark_drift_region(drift_start_2, drift_end_2)

pred_labels = np.zeros(num_batches, dtype=int)
for b in drift_points:
    pred_labels[b] = 1


def confusion_matrix_elements(y_true, y_pred):
    assert len(y_true) == len(y_pred)
    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, FP, TN, FN


TP, FP, TN, FN = confusion_matrix_elements(gt_labels, pred_labels)

if (TP + FN) > 0:
    TPR = TP / (TP + FN)
else:
    TPR = 0.0

if (FP + TN) > 0:
    FPR = FP / (FP + TN)
else:
    FPR = 0.0

print("\n=== SHAP Drift Detection Metrics ===")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"TPR (Recall): {TPR:.4f}")
print(f"FPR: {FPR:.4f}")


# Detection Latency
def compute_latency(start_idx, end_idx):
    start_batch = start_idx // args.batch_size
    end_batch = (end_idx - 1) // args.batch_size
    region_batches = range(start_batch, end_batch + 1)
    detected_batches = [b for b in drift_points if b in region_batches]
    if len(detected_batches) == 0:
        return None
    else:
        first_detected = min(detected_batches)
        return first_detected - start_batch


latency_1 = compute_latency(drift_start_1, drift_end_1)
latency_2 = compute_latency(drift_start_2, drift_end_2)

print(f"Latency for drift region 1: {latency_1} batches")
print(f"Latency for drift region 2: {latency_2} batches")

latencies = [l for l in [latency_1, latency_2] if l is not None]
avg_latency = sum(latencies) / len(latencies) if len(latencies) > 0 else None
print(f"Average latency: {avg_latency} batches")

#################################
# Visualization
#################################
plt.figure(figsize=(12, 6))
plt.plot(shap_scores, label="SHAP Score")
plt.plot(thresholds, label="Dynamic Threshold", linestyle="--")

# Highlight detected drift points
if drift_points:
    plt.scatter(
        drift_points,
        [shap_scores[i] for i in drift_points],
        color="red",
        label="Detected Drifts",
        zorder=5,
    )

# Mark simulated drift regions
plt.axvline(
    x=drift_start_1 // args.batch_size,
    color="orange",
    linestyle="--",
    label="Drift Start 1",
)
plt.axvline(
    x=drift_end_1 // args.batch_size, color="orange", linestyle=":", label="Drift End 1"
)
plt.axvline(
    x=drift_start_2 // args.batch_size,
    color="green",
    linestyle="--",
    label="Drift Start 2",
)
plt.axvline(
    x=drift_end_2 // args.batch_size, color="green", linestyle=":", label="Drift End 2"
)

plt.xlabel("Batch Index")
plt.ylabel("Mean |SHAP Values| (Batch-Level)")
plt.title("SHAP-Based Drift Detection with Multiple True Domain Shifts")
plt.legend()

plot_path = os.path.join(args.output_dir, "shap_drift_detection.png")
plt.savefig(plot_path)
plt.show()
print(f"Plot saved to {plot_path}")